# CampusPulse: bin fill-level detector (YOLO11n)
Fine-tunes YOLO11n on a Roboflow Universe dataset of bins (empty / full / overflow).
Set your Roboflow API key as a Colab secret named `ROBOFLOW_API_KEY` (key icon in the left sidebar).

In [ ]:
!pip -q install ultralytics roboflow

In [ ]:
# Fill these from Roboflow: Download dataset -> YOLOv8 -> show download code
WORKSPACE = "your-workspace"
PROJECT = "your-project"
VERSION = 1

from google.colab import userdata
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
ds = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8")
print(ds.location)

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")
results = model.train(data=f"{ds.location}/data.yaml", epochs=60, imgsz=640, batch=32, patience=15,
                      project="runs", name="bin", exist_ok=True)

In [ ]:
# Validation metrics for the PPT CV slide
best = YOLO("runs/bin/weights/best.pt")
m = best.val(data=f"{ds.location}/data.yaml")
print(f"mAP50: {m.box.map50:.3f}   mAP50-95: {m.box.map:.3f}")
print("classes:", best.names)

In [ ]:
from IPython.display import Image, display
display(Image("runs/bin/confusion_matrix.png", width=600))
display(Image("runs/bin/results.png", width=900))

In [ ]:
import shutil
from google.colab import files
shutil.copy("runs/bin/weights/best.pt", "bin.pt")
files.download("bin.pt")  # copy to backend/models/bin.pt